In [0]:
# RETAIL DELTA PROJECT - SETUP
CATALOG = "retail_demo"

RAW_SCHEMA = f"{CATALOG}.raw"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"
BASE_PATH = "/Volumes/retail_demo/raw/retail_files/retail_delta_project"
BATCH_PATH = f"{BASE_PATH}/datasets/batch"
INCREMENTAL_PATH = f"{BASE_PATH}/datasets/incremental"
print("Catalog:", CATALOG)
print("Base Path:", BASE_PATH)
print("Batch Path:", BATCH_PATH)
print("Incremental Path:", INCREMENTAL_PATH)

Catalog: retail_demo
Base Path: /Volumes/retail_demo/raw/retail_files/retail_delta_project
Batch Path: /Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch
Incremental Path: /Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental


In [0]:
display(dbutils.fs.ls(BASE_PATH))

path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/,_checkpoints/,0,1786555304085
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_schemas/,_schemas/,0,1786555304086
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/checkpoints/,checkpoints/,0,1786555304086
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/,datasets/,0,1786555304086
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/schemas/,schemas/,0,1786555304086


In [0]:
display(dbutils.fs.ls(BATCH_PATH))

path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch/customers_batch.csv,customers_batch.csv,147449,1786421281000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch/orders_batch.csv,orders_batch.csv,1071308,1786421282000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch/products_batch.csv,products_batch.csv,50712,1786421281000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch/stores_batch.csv,stores_batch.csv,2819,1786421281000


In [0]:
display(dbutils.fs.ls(INCREMENTAL_PATH))

path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/,day_2026-04-24/,0,1786555315792
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/,day_2026-04-25/,0,1786555315792
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/,day_2026-04-26/,0,1786555315792


In [0]:
customers_batch = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BATCH_PATH}/customers_batch.csv")
)
display(customers_batch)

customer_id,customer_name,city,segment,gender,signup_date,status
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive


In [0]:
customers_batch.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
orders_batch = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BATCH_PATH}/orders_batch.csv")
)
display(orders_batch.limit(10))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled
O0000006,2026-03-11 02:59:00,C01745,P00101,S019,2,12387.77,0.05,21059.21,COD,returned
O0000007,2025-12-26 03:16:00,C01633,P00555,S020,2,54612.57,0.05,98302.63,COD,returned
O0000008,2025-12-09 20:57:00,C01960,P00771,S029,1,29725.2,0.0,28238.94,UPI,returned
O0000009,2026-01-20 19:25:00,C00479,P00782,S050,4,84092.97,0.05,336371.88,UPI,delivered
O0000010,2026-03-21 06:52:00,C00197,P00068,S049,5,31466.94,0.0,157334.7,COD,delivered


In [0]:
orders_batch.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [0]:
products_batch = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BATCH_PATH}/products_batch.csv")
)
display(products_batch.limit(10))

product_id,product_name,category,brand,unit_price,status,created_date
P00001,T-Shirt 1,Fashion,BrandC,unknown,discontinued,2025-02-15
P00002,Bedsheet 2,Home,BrandC,unknown,active,2024-04-20
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07
P00005,Oil 5,null,BrandC,51796.39,active,2024-12-12
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19
P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23
P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22
P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02


In [0]:
products_batch.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)



In [0]:
stores_batch = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BATCH_PATH}/stores_batch.csv")
)
display(stores_batch.limit(10))

store_id,store_name,city,region,status
S001,Store_1,Jaipur,Online,active
S002,Store_2,Pune,West,active
S003,Store_3,Ahmedabad,South,closed
S004,Store_4,Ahmedabad,North,active
S005,Store_5,Chennai,East,active
S006,Store_6,Kolkata,East,closed
S007,Store_7,Hyderabad,Online,active
S008,Store_8,Chennai,North,closed
S009,Store_9,Bengaluru,West,active
S010,Store_10,Gurugram,East,active


In [0]:
stores_batch.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
print("Customers:", customers_batch.count())
print("Orders:", orders_batch.count())
print("Products:", products_batch.count())
print("Stores:", stores_batch.count())

Customers: 2560
Orders: 12180
Products: 830
Stores: 80
